# Prediccion de paro cardiaco

Notebook basado en `heart_failure.ipynb`, adaptado para usar el dataset `Medicaldataset.csv` de la carpeta `Base-de-datos`.

El flujo conserva la idea principal del notebook original:

1. Importar librerias.
2. Cargar y explorar el dataset.
3. Preparar la variable objetivo `Result`.
4. Escalar variables predictoras.
5. Entrenar modelos supervisados: SVM, KNN y Regresion Logistica.
6. Comparar metricas y revisar el mejor modelo.

En este dataset, `Result` indica si el caso es `positive` o `negative`. Para modelado se codifica como `positive = 1` y `negative = 0`.

## 1. Importar librerias

Se cargan las librerias necesarias para analisis de datos, visualizacion, preprocesamiento, entrenamiento y evaluacion de modelos.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn import preprocessing, svm
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

sns.set_style("whitegrid")
%matplotlib inline

## 2. Cargar el dataset

El notebook original cargaba `heart_failure_clinical_records_dataset.csv.xls`. En este proyecto se usa `Medicaldataset.csv`, ubicado dentro de `Base-de-datos`.

Se dejan dos rutas posibles para que funcione tanto si el notebook se ejecuta desde la carpeta `Supervisado` como desde la raiz del repositorio.

In [2]:
DATA_PATH_CANDIDATES = [
    Path("../Base-de-datos/Medicaldataset.csv"),
    Path("Base-de-datos/Medicaldataset.csv"),
    Path("../Base-de-datos/medicaldatase.csv"),
    Path("Base-de-datos/medicaldatase.csv"),
]

DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No se encontro Medicaldataset.csv en Base-de-datos. Revisa la ruta del archivo.")

data_df = pd.read_csv(DATA_PATH)

print(f"Dataset cargado desde: {DATA_PATH}")
print(f"Filas y columnas: {data_df.shape}")
data_df.head()

FileNotFoundError: No se encontro Medicaldataset.csv en Base-de-datos. Revisa la ruta del archivo.

## 3. Exploracion inicial

Se revisan tipos de datos, estadisticas descriptivas, valores faltantes y distribucion de la variable objetivo.

In [ ]:
data_df.info()

In [ ]:
data_df.describe().T

In [ ]:
print("Valores faltantes por columna:")
display(data_df.isnull().sum())

print("\nDistribucion de Result:")
display(data_df["Result"].value_counts())

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=data_df, x="Result", palette="Set1")
ax.bar_label(ax.containers[0])
plt.title("Distribucion de la variable objetivo")
plt.xlabel("Resultado")
plt.ylabel("Cantidad de pacientes")
plt.show()

## 4. Preparar la variable objetivo

Los modelos de scikit-learn necesitan una variable objetivo numerica. Por eso se transforma `Result` asi:

- `negative` -> `0`
- `positive` -> `1`

In [ ]:
data_model = data_df.copy()
data_model["Result"] = data_model["Result"].str.strip().str.lower()
data_model["Result"] = data_model["Result"].map({"negative": 0, "positive": 1})

if data_model["Result"].isnull().any():
    invalid_values = data_df.loc[data_model["Result"].isnull(), "Result"].unique()
    raise ValueError(f"Valores no reconocidos en Result: {invalid_values}")

data_model.head()

## 5. Analisis bivariado

Se revisa la matriz de correlacion ya con `Result` codificado. Esto ayuda a observar que variables tienen mayor relacion lineal con el resultado.

In [ ]:
plt.figure(figsize=(12, 9))
corrmat = data_model.corr(numeric_only=True)
sns.heatmap(corrmat, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Matriz de correlacion")
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
ax = sns.countplot(data=data_model, x="Age", hue="Result", palette="Set1")
ax.set_title("Distribucion de edad por resultado")
ax.set_xlabel("Edad")
ax.set_ylabel("Cantidad de pacientes")
plt.xticks(rotation=90)
plt.legend(title="Result", labels=["Negative", "Positive"])
plt.show()

## 6. Revision visual de posibles valores extremos

Igual que en el notebook base, se revisan variables no binarias con graficas tipo boxplot para detectar dispersion y posibles outliers.

In [ ]:
numeric_features = [
    "Age",
    "Heart rate",
    "Systolic blood pressure",
    "Diastolic blood pressure",
    "Blood sugar",
    "CK-MB",
    "Troponin",
]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for index, feature in enumerate(numeric_features):
    sns.boxplot(data=data_model, x="Result", y=feature, ax=axes[index], palette="Set1")
    axes[index].set_title(feature)
    axes[index].set_xlabel("Result: 0 = Negative, 1 = Positive")

for index in range(len(numeric_features), len(axes)):
    axes[index].axis("off")

plt.tight_layout()
plt.show()

## 7. Separar variables predictoras y variable objetivo

- `X`: variables clinicas usadas para predecir.
- `y`: variable objetivo `Result` ya codificada.

In [ ]:
X = data_model.drop(["Result"], axis=1)
y = data_model["Result"]

print(f"X: {X.shape}")
print(f"y: {y.shape}")
X.head()

## 8. Escalar variables

SVM, KNN y Regresion Logistica son sensibles a la escala de las variables. Por eso se aplica `StandardScaler` antes de entrenar.

In [ ]:
col_names = list(X.columns)

s_scaler = preprocessing.StandardScaler()
X_scaled = s_scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=col_names)

X_scaled.describe().T

## 9. Separar entrenamiento y prueba

Se conserva una division similar al notebook original: 70% entrenamiento y 30% prueba. Se agrega `stratify=y` para mantener la proporcion de clases en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.30,
    random_state=25,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test : {y_test.shape}")

## 10. Funcion de evaluacion

Para evitar repetir codigo, se define una funcion que entrena cada modelo y calcula accuracy, precision, recall, F1-score y ROC-AUC.

In [ ]:
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    y_proba = None
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        decision = model.decision_function(X_test)
        min_decision = decision.min()
        max_decision = decision.max()
        if max_decision != min_decision:
            y_proba = (decision - min_decision) / (max_decision - min_decision)

    metrics = {
        "Modelo": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    }

    print(f"Evaluation for {model_name}".center(75, "_"))
    for metric, value in metrics.items():
        if metric != "Modelo":
            print(f"{metric}: {value:.4f}")

    return model, y_pred, y_proba, metrics

# Model Building

## 11. Support Vector Machine

In [ ]:
svm_model = svm.SVC(kernel="linear", probability=True, random_state=25)

svm_model, svm_pred, svm_proba, svm_metrics = evaluate_model(
    "Support Vector Machine",
    svm_model,
    X_train,
    X_test,
    y_train,
    y_test,
)

In [ ]:
print("Confusion Matrix For Support Vector Machine:")
print(confusion_matrix(y_test, svm_pred))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    svm_pred,
    display_labels=["Negative", "Positive"],
    cmap="Blues",
)
plt.title("Matriz de confusion - Support Vector Machine")
plt.show()

In [ ]:
print("Classification Report For Support Vector Machine:")
print(classification_report(y_test, svm_pred, target_names=["Negative", "Positive"]))

## 12. KNeighborsClassifier

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model, knn_pred, knn_proba, knn_metrics = evaluate_model(
    "K-Nearest Neighbors",
    knn_model,
    X_train,
    X_test,
    y_train,
    y_test,
)

In [ ]:
print("Confusion Matrix For K-Nearest Neighbors:")
print(confusion_matrix(y_test, knn_pred))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    knn_pred,
    display_labels=["Negative", "Positive"],
    cmap="Blues",
)
plt.title("Matriz de confusion - K-Nearest Neighbors")
plt.show()

In [ ]:
print("Classification Report For K-Nearest Neighbors:")
print(classification_report(y_test, knn_pred, target_names=["Negative", "Positive"]))

## 13. Logistic Regression

In [ ]:
log_model = LogisticRegression(max_iter=1000, random_state=25)

log_model, log_pred, log_proba, log_metrics = evaluate_model(
    "Logistic Regression",
    log_model,
    X_train,
    X_test,
    y_train,
    y_test,
)

In [ ]:
print("Confusion Matrix For Logistic Regression:")
print(confusion_matrix(y_test, log_pred))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    log_pred,
    display_labels=["Negative", "Positive"],
    cmap="Blues",
)
plt.title("Matriz de confusion - Logistic Regression")
plt.show()

In [ ]:
print("Classification Report For Logistic Regression:")
print(classification_report(y_test, log_pred, target_names=["Negative", "Positive"]))

# Comparacion

In [ ]:
metrics_df = pd.DataFrame([svm_metrics, knn_metrics, log_metrics])
metrics_df = metrics_df.set_index("Modelo").sort_values(by="F1", ascending=False)
metrics_df

In [ ]:
metrics_df[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].plot(
    kind="bar",
    figsize=(11, 5),
    ylim=(0, 1),
)
plt.title("Comparacion de modelos")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 14. Curvas ROC

La curva ROC permite comparar visualmente la capacidad de separacion de los modelos entre casos positivos y negativos.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

probas = {
    "Support Vector Machine": svm_proba,
    "K-Nearest Neighbors": knn_proba,
    "Logistic Regression": log_proba,
}

for model_name, y_proba in probas.items():
    if y_proba is not None:
        RocCurveDisplay.from_predictions(y_test, y_proba, name=model_name, ax=ax)

plt.title("Curvas ROC por modelo")
plt.show()

## 15. Mejor modelo segun F1-score

En problemas medicos no basta con ver accuracy. El F1-score balancea precision y recall, por eso se usa aqui para seleccionar el mejor modelo de la comparacion.

In [ ]:
best_model_name = metrics_df.index[0]
best_f1 = metrics_df.loc[best_model_name, "F1"]

print(f"Mejor modelo segun F1-score: {best_model_name}")
print(f"F1-score: {best_f1:.4f}")

## 16. Prediccion de ejemplo

Se toma un paciente del conjunto de prueba y se genera una prediccion individual con el mejor modelo.

In [ ]:
models = {
    "Support Vector Machine": svm_model,
    "K-Nearest Neighbors": knn_model,
    "Logistic Regression": log_model,
}

best_model = models[best_model_name]
sample = X_test.iloc[[0]]

prediction = best_model.predict(sample)[0]
probability = best_model.predict_proba(sample)[0]

print("Prediccion:", "Positive" if prediction == 1 else "Negative")
print(f"P(Negative): {probability[0]:.3f}")
print(f"P(Positive): {probability[1]:.3f}")

## 17. Conclusion

En este notebook se adapto el flujo de `heart_failure.ipynb` al dataset `Medicaldataset.csv`.

Los principales cambios fueron:

1. Se cambio la carga de datos para usar `Base-de-datos/Medicaldataset.csv`.
2. Se uso `Result` como variable objetivo en lugar de `DEATH_EVENT`.
3. Se codifico `Result` como `negative = 0` y `positive = 1`.
4. Se mantuvo la comparacion entre SVM, KNN y Regresion Logistica.
5. Se agregaron metricas comparables para seleccionar el mejor modelo por F1-score.

Este notebook corresponde a un problema de aprendizaje supervisado, ya que aprende patrones a partir de variables clinicas y una etiqueta conocida.

## 18. Guardar modelo entrenado para la app

Al ejecutar esta celda, el mejor modelo encontrado queda guardado en `models/` con `joblib` para poder cargarlo despues desde la aplicacion.

En este caso tambien se guarda el `scaler`, porque el modelo fue entrenado con variables escaladas.

In [ ]:
import json
import joblib

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

trained_models = {
    "Support Vector Machine": svm_model,
    "K-Nearest Neighbors": knn_model,
    "Logistic Regression": log_model,
}

best_model = trained_models[best_model_name]
metrics_by_model = {
    model_name: {metric: float(value) for metric, value in values.items()}
    for model_name, values in metrics_df.to_dict(orient="index").items()
}

paro_metadata = {
    "model_name": best_model_name,
    "source_dataset": "Base-de-datos/Medicaldataset.csv",
    "target_column": "Result",
    "class_labels": {"0": "Negative", "1": "Positive"},
    "feature_columns": list(X.columns),
    "label_mapping": {"negative": 0, "positive": 1},
    "test_metrics_by_model": metrics_by_model,
}

paro_bundle = {
    "model": best_model,
    "scaler": s_scaler,
    "metadata": paro_metadata,
}

joblib.dump(best_model, MODELS_DIR / "paro_cardiaco_best_model.joblib")
joblib.dump(s_scaler, MODELS_DIR / "paro_cardiaco_scaler.joblib")
joblib.dump(paro_bundle, MODELS_DIR / "paro_cardiaco_model.joblib")

with open(MODELS_DIR / "paro_cardiaco_metadata.json", "w", encoding="utf-8") as file:
    json.dump(paro_metadata, file, indent=2)

print("Modelo de paro cardiaco guardado en:")
print(MODELS_DIR / "paro_cardiaco_best_model.joblib")
print(MODELS_DIR / "paro_cardiaco_scaler.joblib")
print(MODELS_DIR / "paro_cardiaco_model.joblib")